An end-to-end NLP project exploring language modelling, text generation, and conversational AI.

In [2]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()
nested_project_dir = PROJECT_DIR / "Deep_Learning" / "Transformer_Language_model"
if (nested_project_dir / "data" / "raw").is_dir():
    PROJECT_DIR = nested_project_dir

while PROJECT_DIR != PROJECT_DIR.parent and not (PROJECT_DIR / "data" / "raw").is_dir():
    PROJECT_DIR = PROJECT_DIR.parent

if not (PROJECT_DIR / "data" / "raw").is_dir():
    raise FileNotFoundError("Could not locate the Transformer_Language_model project directory")


Import all three datasets

In [3]:
DATA_DIR = PROJECT_DIR / "data" / "raw"

girl_df = pd.read_csv(DATA_DIR / "girl_only.csv")

boy_df = pd.read_csv(DATA_DIR / "boy_only.csv")

mixed_df = pd.read_csv(DATA_DIR / "mixed.csv")


In [4]:
print("Girl dataset:", girl_df.shape)
print("Boy dataset:", boy_df.shape)
print("Mixed dataset:", mixed_df.shape)

Girl dataset: (58423, 6)
Boy dataset: (64646, 6)
Mixed dataset: (123069, 6)


In [5]:
# girl_df.head()

In [6]:
# boy_df.head(5)

In [7]:
datasets = {
    "Girl": girl_df,
    "Boy": boy_df,
    "Mixed": mixed_df
}

for name, data in datasets.items():
    print(f"\n{name} Dataset")
    print("-" * 30)
    print("Shape:", data.shape)
    print("Columns:", list(data.columns))
    print("Missing values:", data["text"].isna().sum())


Girl Dataset
------------------------------
Shape: (58423, 6)
Columns: ['timestamp_normalized', 'timestamp_raw', 'speaker', 'text', 'source', 'sender_raw']
Missing values: 2800

Boy Dataset
------------------------------
Shape: (64646, 6)
Columns: ['timestamp_normalized', 'timestamp_raw', 'speaker', 'text', 'source', 'sender_raw']
Missing values: 1332

Mixed Dataset
------------------------------
Shape: (123069, 6)
Columns: ['timestamp_normalized', 'timestamp_raw', 'speaker', 'text', 'source', 'sender_raw']
Missing values: 4132


In [8]:

# print("Speaker distribution:")
# print(mixed_df["speaker"].value_counts())

# print("\nUnique speakers:")
# print(mixed_df["speaker"].unique())

In [9]:
text = mixed_df["text"].fillna("").astype(str)

print("Empty messages:", (text.str.strip() == "").sum())

print("Deleted messages:",
      text.str.contains("deleted this message", case=False).sum())

print("Media placeholders:",
      text.str.contains("<Media omitted>", case=False).sum())

Empty messages: 4132
Deleted messages: 220
Media placeholders: 549


Duplicate


In [10]:
# Check exact duplicate records
duplicate_mask = mixed_df.duplicated(
    subset=["timestamp_normalized", "speaker", "text"],
    keep=False
)

duplicates = mixed_df[duplicate_mask]

print("Total duplicate records:", len(duplicates))

print("Unique duplicate groups:",
      duplicates.groupby(
          ["timestamp_normalized", "speaker", "text"]
      ).ngroups)

Total duplicate records: 4795
Unique duplicate groups: 1824


In [11]:
cross_source = (
    mixed_df.groupby(
        ["timestamp_normalized", "speaker", "text"]
    )
    .agg(
        source_count=("source", "nunique"),
        sources=("source", lambda x: list(x.unique())),
        occurrences=("source", "size")
    )
    .reset_index()
)

cross_source_duplicates = cross_source[
    cross_source["source_count"] > 1
]

print("Cross-source duplicate groups:",
      len(cross_source_duplicates))

cross_source_duplicates.head(10)

Cross-source duplicate groups: 0


,timestamp_normalized,speaker,text,source_count,sources,occurrences


In [12]:
# repeated_messages = (
#     mixed_df.groupby(["speaker", "text"])
#     .agg(
#         occurrences=("text", "size"),
#         unique_timestamps=("timestamp_normalized", "nunique")
#     )
#     .reset_index()
# )

# repeated_messages = repeated_messages[
#     repeated_messages["occurrences"] > 1
# ]

# print("Repeated message groups:", len(repeated_messages))

# repeated_messages.sort_values(
#     "occurrences",
#     ascending=False
# ).head(15)

Removing Duplicate ones ---------

Duplicate-flagged records: 4,795

Duplicate groups: 1,824

Extra records to remove: 2,971

In [13]:
# Create a copy of the original dataset
mixed_clean = mixed_df.copy()

# Count records that will be removed
duplicate_count = mixed_clean.duplicated(
    subset=["timestamp_normalized", "speaker", "text"]
).sum()

print("Records to remove:", duplicate_count)

# Remove exact duplicates, keeping the first occurrence
mixed_clean = mixed_clean.drop_duplicates(
    subset=["timestamp_normalized", "speaker", "text"],
    keep="first"
).reset_index(drop=True)

print("Original records:", len(mixed_df))
print("Records after deduplication:", len(mixed_clean))
print("Records removed:", duplicate_count)

Records to remove: 2682
Original records: 123069
Records after deduplication: 120387
Records removed: 2682


In [14]:
remaining_duplicates = mixed_clean.duplicated(
    subset=["timestamp_normalized", "speaker", "text"]
).sum()

print("Remaining exact duplicates:", remaining_duplicates)

Remaining exact duplicates: 0


will apply same logic to all three datasets 

In [15]:
def remove_exact_duplicates(df):
    return df.drop_duplicates(
        subset=["timestamp_normalized", "speaker", "text"],
        keep="first"
    ).reset_index(drop=True)


girl_clean = remove_exact_duplicates(girl_df)
boy_clean = remove_exact_duplicates(boy_df)
mixed_clean = remove_exact_duplicates(mixed_df)

In [16]:
datasets_before = {
    "Girl": girl_df,
    "Boy": boy_df,
    "Mixed": mixed_df
}

datasets_after = {
    "Girl": girl_clean,
    "Boy": boy_clean,
    "Mixed": mixed_clean
}

for name in datasets_before:
    before = len(datasets_before[name])
    after = len(datasets_after[name])

    print(f"\n{name} Dataset")
    print("Before:", before)
    print("After:", after)
    print("Removed:", before - after)


Girl Dataset
Before: 58423
After: 57475
Removed: 948

Boy Dataset
Before: 64646
After: 62912
Removed: 1734

Mixed Dataset
Before: 123069
After: 120387
Removed: 2682


In [17]:
for name, df in datasets_after.items():
    remaining = df.duplicated(
        subset=["timestamp_normalized", "speaker", "text"]
    ).sum()

    print(name, "remaining duplicates:", remaining)

Girl remaining duplicates: 0
Boy remaining duplicates: 0
Mixed remaining duplicates: 0


Duplicates are finally removed , moving to chronological ordering 

In [18]:
for name, df in datasets_after.items():

    timestamps = pd.to_datetime(
        df["timestamp_normalized"],
        errors="coerce"
    )

    print(f"\n{name} Dataset")
    print("Invalid timestamps:", timestamps.isna().sum())
    print("Earliest:", timestamps.min())
    print("Latest:", timestamps.max())
    print("Chronologically sorted:",
          timestamps.is_monotonic_increasing)


Girl Dataset
Invalid timestamps: 0
Earliest: 2025-01-26 12:32:00
Latest: 2026-09-22 08:36:00
Chronologically sorted: True

Boy Dataset
Invalid timestamps: 0
Earliest: 2025-01-26 11:56:00
Latest: 2026-09-22 09:14:00
Chronologically sorted: True

Mixed Dataset
Invalid timestamps: 0
Earliest: 2025-01-26 11:56:00
Latest: 2026-09-22 09:14:00
Chronologically sorted: True


In [19]:
def classify_message(df):

    df = df.copy()

    text = df["text"].fillna("").astype(str).str.strip()

    conditions = [
        text.eq(""),
        text.str.contains("deleted this message", case=False, na=False),
        text.str.contains("<Media omitted>", case=False, na=False)
    ]

    categories = [
        "empty",
        "deleted",
        "media",
    ]

    df["message_type"] = np.select(
        conditions,
        categories,
        default="text"
    )

    return df


girl_clean = classify_message(girl_clean)
boy_clean = classify_message(boy_clean)
mixed_clean = classify_message(mixed_clean)

In [20]:
for name, df in {
    "Girl": girl_clean,
    "Boy": boy_clean,
    "Mixed": mixed_clean
}.items():

    print(f"\n{name} Dataset")
    print(df["message_type"].value_counts())


Girl Dataset
message_type
text     54752
empty     2580
media      143
Name: count, dtype: int64

Boy Dataset
message_type
text       61307
empty       1226
media        228
deleted      151
Name: count, dtype: int64

Mixed Dataset
message_type
text       116059
empty        3806
media         371
deleted       151
Name: count, dtype: int64


In [21]:
for name, df in {
    "Girl": girl_clean,
    "Boy": boy_clean,
    "Mixed": mixed_clean
}.items():

    usable = df[df["message_type"] == "text"]

    print(f"\n{name}")
    print("Total records:", len(df))
    print("Usable text records:", len(usable))
    print("Non-text records:", len(df) - len(usable))


Girl
Total records: 57475
Usable text records: 54752
Non-text records: 2723

Boy
Total records: 62912
Usable text records: 61307
Non-text records: 1605

Mixed
Total records: 120387
Usable text records: 116059
Non-text records: 4328


Inspect empty and deleted messages 

In [22]:
# empty_records = mixed_clean[
#     mixed_clean["message_type"] == "empty"
# ]

# display(
#     empty_records[
#         ["timestamp_normalized", "speaker", "text", "source"]
#     ].head(15)
# )

In [23]:
# deleted_records = mixed_clean[
#     mixed_clean["message_type"] == "deleted"
# ]

# display(
#     deleted_records[
#         ["timestamp_normalized", "speaker", "text", "source"]
#     ].head(15)
# )

Removing the empty records

In [24]:
girl_clean = girl_clean[
    girl_clean["message_type"] != "empty"
].reset_index(drop=True)

boy_clean = boy_clean[
    boy_clean["message_type"] != "empty"
].reset_index(drop=True)

mixed_clean = mixed_clean[
    mixed_clean["message_type"] != "empty"
].reset_index(drop=True)

In [25]:
for name, df in {
    "Girl": girl_clean,
    "Boy": boy_clean,
    "Mixed": mixed_clean
}.items():

    print(f"\n{name} Dataset")
    print("Total records:", len(df))
    print("Text:", (df["message_type"] == "text").sum())
    print("Media:", (df["message_type"] == "media").sum())
    print("Deleted:", (df["message_type"] == "deleted").sum())
    print("Empty:", (df["message_type"] == "empty").sum())


Girl Dataset
Total records: 54895
Text: 54752
Media: 143
Deleted: 0
Empty: 0

Boy Dataset
Total records: 61686
Text: 61307
Media: 228
Deleted: 151
Empty: 0

Mixed Dataset
Total records: 116581
Text: 116059
Media: 371
Deleted: 151
Empty: 0


Text normalization and tokenization 

In [26]:
girl_text = girl_clean[
    girl_clean["message_type"] == "text"
].copy()

boy_text = boy_clean[
    boy_clean["message_type"] == "text"
].copy()

mixed_text = mixed_clean[
    mixed_clean["message_type"] == "text"
].copy()

print("Girl text messages:", len(girl_text))
print("Boy text messages:", len(boy_text))
print("Mixed text messages:", len(mixed_text))


Girl text messages: 54752
Boy text messages: 61307
Mixed text messages: 116059


In [27]:
def normalize_text(text):

    text = str(text)

    # Remove leading and trailing whitespace
    text = text.strip()

    # Replace multiple spaces with a single space
    text = " ".join(text.split())

    return text


girl_text["text_normalized"] = girl_text["text"].apply(normalize_text)

print(girl_text[["text", "text_normalized"]].head(15))

                          text             text_normalized
0                 Koi baat nhi                Koi baat nhi
1      Tumne msg kya kiya tha?     Tumne msg kya kiya tha?
2         Happy Republic Day 😊        Happy Republic Day 😊
3                          Btw                         Btw
4                    Same to u                   Same to u
5                Aaj jana tha?               Aaj jana tha?
6                 Ohh bhul gyi                Ohh bhul gyi
7            Kal chali jayungi           Kal chali jayungi
8                       Acha ?                      Acha ?
9           Konsi country ka ?          Konsi country ka ?
10  Btw thanks for the treat 😁  Btw thanks for the treat 😁
11              Kal Monday hai              Kal Monday hai
12           Samajh nhi aaya 😅           Samajh nhi aaya 😅
13                         Nhi                         Nhi
14    Chocolate mili thi parso    Chocolate mili thi parso


In [28]:
from collections import Counter

all_text = " ".join(girl_text["text_normalized"])

# Split into word-level tokens
tokens = all_text.split()

# Count token frequencies
token_counts = Counter(tokens)

# Create vocabulary
vocab = sorted(token_counts.keys())

# Assign numerical IDs
stoi = {token: idx for idx, token in enumerate(vocab)}
itos = {idx: token for token, idx in stoi.items()}

print("Total tokens:", len(tokens))
print("Vocabulary size:", len(vocab))
print("Most common tokens:", token_counts.most_common(20))

Total tokens: 274553
Vocabulary size: 17374
Most common tokens: [('hai', 12794), ('nhi', 7369), ('to', 6815), ('bhi', 3954), ('me', 3051), ('se', 2868), ('mai', 2804), ('hu', 2777), ('ke', 2728), ('hi', 2704), ('aur', 2519), ('kya', 2461), ('na', 2342), ('tu', 2324), ('ho', 2242), ('kar', 2232), ('rha', 2202), ('?', 2168), ('baat', 2079), ('rhi', 1987)]


In [29]:
def encode(text):
    tokens = text.split()
    return [stoi[token] for token in tokens if token in stoi]


def decode(ids):
    return " ".join(itos[idx] for idx in ids)


sample = girl_text["text_normalized"].iloc[0]

encoded = encode(sample)

print("Original:", sample)
print("Encoded:", encoded)
print("Decoded:", decode(encoded))

Original: Koi baat nhi
Encoded: [2351, 4882, 12487]
Decoded: Koi baat nhi


Implementing BPE tokenizer to handle unseen text that can appear during training 

In [37]:
# !pip install tokenizers

In [31]:
# Keep only valid text messages
messages = girl_text["text_normalized"].dropna()

messages = messages[
    messages.str.strip() != ""
].tolist()

# Preserve chronological order
split_index = int(len(messages) * 0.90)

train_messages = messages[:split_index]
val_messages = messages[split_index:]

print("Training messages:", len(train_messages))
print("Validation messages:", len(val_messages))

Training messages: 49276
Validation messages: 5476


Split data

In [32]:
from tokenizers import ByteLevelBPETokenizer

tokenizer = ByteLevelBPETokenizer()

special_tokens = [
    "<|unk|>",
    "<|pad|>",
    "<|bos|>",
    "<|eos|>"
]

tokenizer.train_from_iterator(
    train_messages,
    vocab_size=2000,
    min_frequency=2,
    special_tokens=special_tokens,
    show_progress=True
)

print("BPE training completed!")
print("Vocabulary size:", tokenizer.get_vocab_size())

BPE training completed!
Vocabulary size: 2000


In [33]:
sample = "I really love machine learning ❤️"

encoded = tokenizer.encode(sample)

print("Original text:", sample)
print("Token IDs:", encoded.ids)
print("Tokens:", encoded.tokens)
print("Decoded:", tokenizer.decode(encoded.ids))

Original text: I really love machine learning ❤️
Token IDs: [44, 523, 981, 524, 371, 1359, 289, 377, 266, 733, 224, 1214]
Tokens: ['I', 'Ġre', 'ally', 'Ġlove', 'Ġma', 'chi', 'ne', 'Ġle', 'ar', 'ning', 'Ġ', 'âĿ¤ï¸ı']
Decoded: I really love machine learning ❤️


In [34]:
for message in train_messages[:10]:

    output = tokenizer.encode(message)

    print("\nOriginal:", message)
    print("Tokens:", output.tokens)
    print("Token IDs:", output.ids)


Original: Koi baat nhi
Tokens: ['Koi', 'Ġbaat', 'Ġnhi']
Token IDs: [760, 346, 286]

Original: Tumne msg kya kiya tha?
Tokens: ['Tum', 'ne', 'Ġmsg', 'Ġkya', 'Ġkiya', 'Ġtha', '?']
Token IDs: [1598, 289, 617, 347, 503, 343, 34]

Original: Happy Republic Day 😊
Tokens: ['Ha', 'pp', 'y', 'Ġ', 'Re', 'p', 'ub', 'li', 'c', 'Ġ', 'D', 'ay', 'ĠðŁĺĬ']
Token IDs: [446, 862, 92, 224, 1909, 83, 1363, 391, 70, 224, 39, 979, 1505]

Original: Btw
Tokens: ['B', 't', 'w']
Token IDs: [37, 87, 90]

Original: Same to u
Tokens: ['Sa', 'me', 'Ġto', 'Ġu']
Token IDs: [772, 338, 287, 321]

Original: Aaj jana tha?
Tokens: ['Aaj', 'Ġjana', 'Ġtha', '?']
Token IDs: [766, 652, 343, 34]

Original: Ohh bhul gyi
Tokens: ['Ohh', 'Ġbhu', 'l', 'Ġgyi']
Token IDs: [1572, 1635, 79, 488]

Original: Kal chali jayungi
Tokens: ['Kal', 'Ġchali', 'Ġja', 'y', 'ungi']
Token IDs: [789, 1205, 312, 92, 357]

Original: Acha ?
Tokens: ['Acha', 'Ġ?']
Token IDs: [1587, 307]

Original: Konsi country ka ?
Tokens: ['Kon', 'si', 'Ġco', 'unt', 'r

In [35]:
save_path = PROJECT_DIR / "models" / "krish_bpe"

save_path.mkdir(parents=True, exist_ok=True)

tokenizer.save_model(str(save_path))

print("Tokenizer saved successfully!")

Tokenizer saved successfully!


In [36]:
sample_messages = val_messages[:1000]

word_token_count = sum(
    len(message.split())
    for message in sample_messages
)

bpe_token_count = sum(
    len(tokenizer.encode(message).ids)
    for message in sample_messages
)

print("Word-level tokens:", word_token_count)
print("BPE tokens:", bpe_token_count)

print(
    "BPE / word-level ratio:",
    round(bpe_token_count / word_token_count, 3)
)

Word-level tokens: 4113
BPE tokens: 6495
BPE / word-level ratio: 1.579


### Now we will do Model Training in new jupyter Notebook file token_training 
